In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_parquet("mimic_com_labels_exames_fill.parquet")
df.head(10)

,subject_id,stay_id,janela_index,inicio_janela,charttime,fc,pas,pad,pam,fr,...,ph,uri,hem,ida,pes,alt,label_qsofa,label_mews,label_sofa,tem_sepse
0,10000032,39553978,0,2180-07-23 14:00:00,2180-07-23 14:11:00,91.0,84.0,48.0,56.0,24.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,0,0,0,0.0
1,10000032,39553978,0,2180-07-23 14:00:00,2180-07-23 14:12:00,91.0,84.0,48.0,56.0,24.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,0,0,0,0.0
2,10000032,39553978,0,2180-07-23 14:00:00,2180-07-23 14:13:00,91.0,84.0,48.0,56.0,24.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,0,0,0,0.0
3,10000032,39553978,0,2180-07-23 14:00:00,2180-07-23 14:30:00,93.0,95.0,59.0,67.0,21.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,0,0,0,0.0
4,10000032,39553978,1,2180-07-23 15:00:00,2180-07-23 15:00:00,94.0,88.0,56.0,64.0,23.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,1,1,0,0.0
5,10000032,39553978,2,2180-07-23 16:00:00,2180-07-23 16:00:00,105.0,88.0,56.0,64.0,21.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,0,0,0,0.0
6,10000032,39553978,2,2180-07-23 16:00:00,2180-07-23 16:01:00,105.0,91.0,55.0,64.0,21.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,0,0,0,0.0
7,10000032,39553978,3,2180-07-23 17:00:00,2180-07-23 17:00:00,97.0,95.0,58.0,67.0,20.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,0,0,0,0.0
8,10000032,39553978,4,2180-07-23 18:00:00,2180-07-23 18:00:00,100.0,86.0,53.0,60.0,21.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,0,1,0,0.0
9,10000032,39553978,5,2180-07-23 19:00:00,2180-07-23 19:00:00,97.0,93.0,41.0,56.0,16.0,...,7.41,71.0,34.799999,52.0,39.400002,152.0,0,0,0,0.0


In [3]:
# ===== 1. GARANTIR FORMATO CORRETO =====
df["inicio_janela"] = pd.to_datetime(df["inicio_janela"])
df = df.sort_values(["subject_id", "inicio_janela"])

# ===== 2. DEFINIR FEATURES =====
features = ['fc', 'pas', 'pad', 'pam', 'fr', 'spo', 'tem', 'cre', 'lac', 'leu',
            'pla', 'ph', 'uri', 'hem', 'ida', 'pes', 'alt', 'label_qsofa',
            'label_mews', 'label_sofa']

# ===== 3. FUNÇÃO PARA CRIAR JANELAS =====
def criar_janelas(df, window_size=10):
    X, y, mask = [], [], []

    for pid, grupo in df.groupby("subject_id"):
        grupo = grupo.sort_values("inicio_janela")

        dados = grupo[features].values
        labels = grupo["tem_sepse"].values

        # percorre em blocos de 10 (sem sobreposição)
        for i in range(0, len(grupo), window_size):
            janela = dados[i:i+window_size]
            label_janela = labels[i:i+window_size]

            # padding se necessário
            if len(janela) < window_size:
                pad_len = window_size - len(janela)
                janela = np.pad(janela, ((0, pad_len), (0, 0)), mode='constant')
                mask_janela = [1]*len(label_janela) + [0]*pad_len
            else:
                mask_janela = [1]*window_size

            X.append(janela)
            y.append(int(label_janela.max()))  # se houver sepse na janela → 1
            mask.append(mask_janela)

    return np.array(X), np.array(y), np.array(mask)

# ===== 4. GERAR OS ARRAYS =====
X, y, mask = criar_janelas(df, window_size=10)

# ===== 5. VERIFICAÇÃO =====
print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("Shape mask:", mask.shape)
print("Classes:", np.unique(y))

# ===== 6. SALVAR =====
np.savez("mimic_X_y_mask_10j.npz", X=X, y=y, mask=mask)

print("✅ Arquivo salvo como mimic_X_y_mask_10j.npz")

Shape X: (722017, 10, 20)
Shape y: (722017,)
Shape mask: (722017, 10)
Classes: [0 1]
✅ Arquivo salvo como mimic_X_y_mask_10j.npz
